# 02 · Perfilado de tokenización

Qué límites reales imponen los tokenizadores de los modelos candidatos sobre el input de
embedding, y **por qué se prohíbe el truncamiento silencioso**. Este notebook **consume** el
reporte de `scripts/profile_tokenizers.py`; no descarga modelos ni tokeniza aquí.

Genera el reporte (en el servidor, con red la primera vez):

```bash
uv run python scripts/profile_tokenizers.py --allow-unpinned-revision
```

In [ ]:
import json
from pathlib import Path

import pandas as pd

REPORT = Path("data/processed/reports/tokenizer_profile.json")
if not REPORT.is_file():
    raise FileNotFoundError(
        f"Falta {REPORT}. Ejecuta antes scripts/profile_tokenizers.py (ver docstring del notebook)."
    )
report = json.loads(REPORT.read_text(encoding="utf-8"))
print(f"chunks perfilados: {report['n_chunks']} | modelos: {report['n_models']}")

In [ ]:
rows = []
for m in report["models"]:
    emb = m["embedding_input_profile"]["overall"]
    rows.append(
        {
            "model_id": m["model_id"],
            "effective_max_tokens": m["effective_max_tokens"],
            "source": m["source_of_effective_limit"],
            "n_truncated": emb["n_truncated"],
            "pct_truncated": emb["pct_truncated"],
            "max_tokens": emb["max_tokens"],
            "p95": emb["p95_tokens"],
            "p99": emb["p99_tokens"],
        }
    )
df = pd.DataFrame(rows).sort_values("effective_max_tokens")
df

## Lectura

Los modelos de ventana corta (familia e5, 512 tokens) son los únicos con riesgo real de
*overflow* sobre el corpus; los de contexto largo (bge-m3 8192, qwen3 32k) apenas truncan. En
Fase 2 **nunca** se trunca en silencio: el input que excede el límite se **repara** dividiéndolo
en ventanas token-aware con overlap 100 (`src/embeddings/input_preparation.py`), y si quedara
algún overflow sin reparar la generación se detiene (error bloqueante). `pct_truncated` aquí es
diagnóstico previo; tras la reparación, el bundle publicado tiene 0 inputs truncados.